# Задание №3. Построение нейронной сети для синтеза текста (генерация текста на основе LSTM)

Данное задание представляет собой пошаговое руководство по созданию модели генерации текста с использованием рекуррентных нейронных сетей (LSTM). Вы познакомитесь с основными понятиями обработки естественного языка (NLP), научитесь подготавливать текстовый корпус, строить и обучать модель, а также генерировать новый текст на основе заданного начала с использованием различных методов сэмплирования (температура, top‑k, штраф за повторения). Задание адаптировано для выполнения в **Google Colab** (Jupyter Notebook). Все части работы (подготовка данных, обучение, сохранение, загрузка из репозитория и генерация) выполняются в одном ноутбуке.

---

## 1. Теоретическое введение: ключевые понятия NLP для генерации текста

### 1.1. Корпус (Corpus)
Корпус – это структурированная коллекция текстов, используемая для обучения моделей NLP. В задаче генерации текста обычно используется **монолингвальный корпус** – набор текстов на одном языке, на котором модель учится статистике языка, стилю и тематике.

### 1.2. Токенизация (Tokenization)
Процесс разбиения текста на минимальные единицы – **токены**. В генерации текста часто используют токенизацию на уровне слов (word‑level) или символов (character‑level). В этом задании мы будем работать со словами. Для токенизации используем `Tokenizer` из Keras.

### 1.3. Эмбеддинги (Embeddings)
Векторные представления слов. Слой **Embedding** преобразует целочисленные индексы токенов в плотные векторы фиксированной размерности. Эти векторы обучаются вместе с моделью и позволяют улавливать семантические и синтаксические сходства между словами.

### 1.4. Рекуррентные нейронные сети (RNN, LSTM)
RNN – класс сетей, способных обрабатывать последовательности произвольной длины благодаря внутренней памяти. **LSTM (Long Short‑Term Memory)** – разновидность RNN, которая эффективно обучается на длинных последовательностях, избегая проблемы затухания градиента. В генерации текста LSTM используются для предсказания следующего слова на основе предыдущих.

### 1.5. Авторегрессионная генерация текста
Модель обучается предсказывать следующее слово в последовательности. Во время генерации мы подаём начальную фразу (seed), модель предсказывает следующее слово, добавляем его к последовательности, повторяем процесс. Такой подход называется авторегрессией.

### 1.6. Сэмплирование, температура, Top‑K и штраф за повторения
На каждом шаге модель выдаёт вероятности для всех слов словаря. Чтобы получить разнообразный и осмысленный текст, применяют различные методы сэмплирования:
- **Жадный выбор** – всегда берётся слово с максимальной вероятностью. Текст становится предсказуемым и часто зацикливается.
- **Сэмплирование с температурой (temperature sampling)**: вероятности модифицируются путём деления логарифмов на температуру `T`. При `T` → 0 распределение стремится к «argmax», при `T` → ∞ – к равномерному. Типичные значения `T` = 0.5…1.5.
- **Top‑K sampling**: учитываются только `k` наиболее вероятных слов, их вероятности перенормируются. Это отсекает «хвост» маловероятных слов.
- **Штраф за повторения (repetition penalty)**: уменьшает вероятность слов, которые уже встречались в сгенерированном тексте, чтобы избежать зацикливания.

---

## 2. Постановка задачи

Вам необходимо создать собственный репозиторий на GitHub, загрузить в него текстовый корпус (или сгенерировать синтетический), затем построить и обучить модель LSTM для генерации текста. После обучения вы сохраните модель и токенизатор, загрузите их в тот же репозиторий. В финальной части ноутбука вы продемонстрируете, как загрузить эти файлы из репозитория и использовать их для генерации текста без повторного обучения. Весь код и отчёт должны быть оформлены в одном Google Colab (Jupyter Notebook).

---

## 3. Структура отчёта

Ваш отчёт должен содержать следующие разделы (в виде ячеек Markdown в ноутбуке):

1. **Титульный лист** (название работы, ФИО, группа, ссылка на репозиторий)
2. **Введение** (цель работы, краткое описание задачи)
3. **Теоретическая часть** (объяснение ключевых понятий: корпус, токенизация, эмбеддинги, LSTM, авторегрессия, сэмплирование с температурой, top‑k, штраф за повторения)
4. **Описание данных** (источник данных, статистика: количество строк, уникальных слов, общая длина текста; ссылка на файл в репозитории)
5. **Подготовка данных** (токенизация, создание обучающих последовательностей, разделение на train/validation)
6. **Построение модели** (архитектура LSTM, визуализация модели, summary)
7. **Обучение модели** (параметры обучения, использование early stopping, графики потерь и точности)
8. **Генерация текста** (описание функции `generate_text`, демонстрация примеров с разными параметрами температуры и top‑k)
9. **Сохранение модели и токенизатора** (код сохранения, загрузка файлов в репозиторий)
10. **Загрузка модели из репозитория и быстрая генерация** (код загрузки через raw‑ссылку, повторное использование для генерации)
11. **Выводы** (что получилось, какие были трудности, возможные улучшения)
12. **Список использованных источников**
13. **Приложение** (полный код с комментариями)

---

## 4. Требования к корпусу и ссылка на репозиторий

### 4.1. Минимальный и максимальный объём корпуса
- **Минимальный объём**: не менее 1000 предложений (чтобы модель могла выучить хотя бы базовые закономерности).
- **Максимальный объём**: в Google Colab ограничение по оперативной памяти (обычно ~12–25 ГБ). Для комфортной работы рекомендуется использовать не более 20 000–30 000 предложений при длине последовательности 5–10 слов. При превышении может потребоваться уменьшить размер словаря или использовать генераторы данных.

### 4.2. Создание репозитория и загрузка корпуса
1. Зарегистрируйтесь на [GitHub](https://github.com) (если у вас ещё нет аккаунта).
2. Создайте новый публичный репозиторий с названием, например, `text-generation-lstm`.
3. Загрузите в репозиторий файл с корпусом (например, `corpus.txt`). Вы можете использовать собственный текстовый файл или сгенерировать синтетические данные с помощью кода ниже.  
   **Важно**: после загрузки получите **raw‑ссылку** на файл. Для этого откройте файл в репозитории, нажмите кнопку "Raw" и скопируйте URL. Он будет выглядеть примерно так:  
   `https://raw.githubusercontent.com/ваш_логин/text-generation-lstm/main/corpus.txt`

### 4.3. Синтетический датасет (если нет реального)
Если у вас нет собственного корпуса, вы можете сгенерировать синтетические данные прямо в ноутбуке и затем сохранить их в файл для загрузки в репозиторий.

```python
import random

def generate_synthetic_corpus(num_sentences=2000):
    subjects = ['кот', 'собака', 'птица', 'рыба', 'хомяк']
    verbs = ['сидит', 'бежит', 'спит', 'ест', 'играет']
    objects = ['на диване', 'в парке', 'в коробке', 'под столом', 'с мячом']
    sentences = []
    for _ in range(num_sentences):
        sent = f"{random.choice(subjects)} {random.choice(verbs)} {random.choice(objects)}"
        sentences.append(sent)
    return sentences

synthetic_sentences = generate_synthetic_corpus(2000)
with open('corpus.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(synthetic_sentences))
```

Затем загрузите файл `corpus.txt` в свой репозиторий.

---

## 5. Пошаговое выполнение задания в Google Colab

### 5.1. Подготовка окружения

Импортируем необходимые библиотеки. В Google Colab они уже предустановлены.

```python
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import pickle
import random
import requests  # для скачивания файлов по ссылке
from sklearn.model_selection import train_test_split
```

### 5.2. Загрузка корпуса из репозитория

Используйте вашу raw‑ссылку на файл `corpus.txt`.

```python
# Вставьте вашу ссылку
url_corpus = "https://raw.githubusercontent.com/ваш_логин/text-generation-lstm/main/corpus.txt"

# Скачиваем файл
response = requests.get(url_corpus)
with open('corpus.txt', 'w', encoding='utf-8') as f:
    f.write(response.text)

# Загружаем корпус
with open('corpus.txt', 'r', encoding='utf-8') as f:
    corpus_lines = [line.strip().lower() for line in f if line.strip()]

print(f"Загружено строк: {len(corpus_lines)}")
print("Пример первой строки:", corpus_lines[0])
```

### 5.3. Токенизация всего текста

Объединим все строки в один большой текст (для создания скользящих окон по всему корпусу). Затем обучим токенизатор на этом тексте.

```python
# Объединяем все строки в один текст
text = ' '.join(corpus_lines)

# Токенизация
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
word_index = tokenizer.word_index
total_words = len(word_index) + 1  # +1 для нулевого токена (padding)

print(f"Размер словаря: {total_words}")
```

### 5.4. Создание обучающих последовательностей

Преобразуем весь текст в последовательность индексов и создаём скользящие окна длины `sequence_length + 1`. Для каждого окна входом будут первые `sequence_length` слов, а целевым – последнее слово.

```python
sequence_length = 5  # длина контекста (сколько предыдущих слов учитываем)
tokens = tokenizer.texts_to_sequences([text])[0]

input_sequences = []
for i in range(sequence_length, len(tokens)):
    input_sequences.append(tokens[i-sequence_length:i+1])

# Преобразуем в numpy массивы
X = np.array([seq[:-1] for seq in input_sequences])  # входные последовательности
y = np.array([seq[-1] for seq in input_sequences])   # целевые слова

print(f"Всего примеров: {len(X)}")
print(f"Форма X: {X.shape}")
print(f"Форма y: {y.shape}")
```

### 5.5. Разделение на обучающую и валидационную выборки

Для контроля переобучения разделим данные на тренировочные (80%) и валидационные (20%).

```python
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Обучающих примеров: {len(X_train)}")
print(f"Валидационных примеров: {len(X_val)}")
```

### 5.6. Построение модели

Используем модель Sequential с Embedding слоем, двумя LSTM слоями (с Dropout для регуляризации) и выходным Dense слоем с softmax.

```python
model = Sequential([
    Embedding(input_dim=total_words, output_dim=128, input_length=sequence_length),
    LSTM(256, return_sequences=True, dropout=0.2),
    LSTM(256, dropout=0.2),
    Dense(total_words, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()
```

### 5.7. Обучение модели

Добавим `EarlyStopping` для предотвращения переобучения и сохраним историю обучения для визуализации.

```python
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=1
)
```

### 5.8. Визуализация процесса обучения

Построим графики потерь и точности на обучающей и валидационной выборках.

```python
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.title('Потери')

plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.legend()
plt.title('Точность')
plt.show()
```

### 5.9. Сохранение модели и токенизатора

Сохраним модель в формате `.h5` и токенизатор с помощью `pickle`. Эти файлы затем нужно загрузить в ваш репозиторий.

```python
# Сохраняем модель
model.save('text_generator.h5')
print("Модель сохранена в text_generator.h5")

# Сохраняем токенизатор
with open('tokenizer.pickle', 'wb') as f:
    pickle.dump(tokenizer, f)
print("Токенизатор сохранён в tokenizer.pickle")
```

### 5.10. Загрузка файлов в репозиторий

1. Скачайте файлы `text_generator.h5` и `tokenizer.pickle` на свой компьютер (через панель файлов Colab).
2. В своём репозитории на GitHub нажмите "Add file" → "Upload files", загрузите оба файла.
3. После загрузки получите raw‑ссылки на каждый файл (аналогично корпусу). Они понадобятся для следующего этапа.

### 5.11. Функция генерации текста (с использованием текущей модели)

Прежде чем перейти к загрузке из репозитория, продемонстрируем работу модели, обучившейся в этом же ноутбуке. Это поможет убедиться, что модель работает.

```python
def generate_text(seed_text, next_words=20, temperature=1.0, top_k=10, rep_penalty=1.2):
    """
    Генерирует текст, начиная с seed_text, используя текущую модель.
    """
    words = seed_text.lower().split()
    generated = words.copy()
    freq = {w: generated.count(w) for w in generated}

    for _ in range(next_words):
        current_seq = ' '.join(generated[-sequence_length:])
        seq = tokenizer.texts_to_sequences([current_seq])[0]
        seq = pad_sequences([seq], maxlen=sequence_length, padding='pre')
        preds = model.predict(seq, verbose=0)[0]

        logits = np.log(preds + 1e-8) / temperature
        exp_logits = np.exp(logits)
        p = exp_logits / exp_logits.sum()

        if top_k is not None and top_k < len(p):
            top_indices = np.argsort(p)[-top_k:]
            mask = np.zeros_like(p)
            mask[top_indices] = 1
            p = p * mask
            p = p / p.sum()

        for idx, word in tokenizer.index_word.items():
            count = freq.get(word, 0)
            if count > 0 and idx < len(p):
                p[idx] = p[idx] / (rep_penalty ** count)
        p = p / p.sum()

        next_idx = np.random.choice(len(p), p=p)
        next_word = tokenizer.index_word.get(next_idx, '')
        if not next_word:
            break
        generated.append(next_word)
        freq[next_word] = freq.get(next_word, 0) + 1

    return ' '.join(generated)

# Пример
print(generate_text("кот сидит", next_words=15, temperature=0.8, top_k=10, rep_penalty=1.5))
```

### 5.12. Загрузка модели и токенизатора из репозитория и быстрая генерация

Этот блок можно выполнить после того, как вы загрузили файлы в репозиторий. Он демонстрирует, как загрузить сохранённые артефакты и использовать их для генерации без повторного обучения.

```python
# Вставьте ваши raw-ссылки
url_model = "https://raw.githubusercontent.com/ваш_логин/text-generation-lstm/main/text_generator.h5"
url_tokenizer = "https://raw.githubusercontent.com/ваш_логин/text-generation-lstm/main/tokenizer.pickle"

# Скачиваем модель (обратите внимание: для .h5 нужна прямая ссылка на скачивание, raw может не работать,
# поэтому лучше использовать ссылку на скачивание с GitHub: https://github.com/.../raw/main/... .h5?raw=true)
# Упростим: можно использовать ссылку вида https://github.com/ваш_логин/text-generation-lstm/raw/main/text_generator.h5
# Для pickle аналогично.

# Фактически GitHub выдаёт файл как есть по ссылке /raw/main/...
url_model = "https://github.com/ваш_логин/text-generation-lstm/raw/main/text_generator.h5"
url_tokenizer = "https://github.com/ваш_логин/text-generation-lstm/raw/main/tokenizer.pickle"

# Скачиваем
!wget -O text_generator.h5 {url_model}
!wget -O tokenizer.pickle {url_tokenizer}

# Загружаем модель и токенизатор
loaded_model = load_model('text_generator.h5')
with open('tokenizer.pickle', 'rb') as f:
    loaded_tokenizer = pickle.load(f)

# Функция генерации с загруженными объектами
def generate_from_loaded(seed_text, next_words=20, temperature=0.8, top_k=10, rep_penalty=1.2):
    words = seed_text.lower().split()
    generated = words.copy()
    freq = {w: generated.count(w) for w in generated}
    seq_len = 5  # должно совпадать

    for _ in range(next_words):
        current_seq = ' '.join(generated[-seq_len:])
        seq = loaded_tokenizer.texts_to_sequences([current_seq])[0]
        seq = pad_sequences([seq], maxlen=seq_len, padding='pre')
        preds = loaded_model.predict(seq, verbose=0)[0]

        logits = np.log(preds + 1e-8) / temperature
        exp_logits = np.exp(logits)
        p = exp_logits / exp_logits.sum()

        if top_k and top_k < len(p):
            top_indices = np.argsort(p)[-top_k:]
            mask = np.zeros_like(p)
            mask[top_indices] = 1
            p = p * mask
            p = p / p.sum()

        for idx, word in loaded_tokenizer.index_word.items():
            count = freq.get(word, 0)
            if count > 0 and idx < len(p):
                p[idx] = p[idx] / (rep_penalty ** count)
        p = p / p.sum()

        next_idx = np.random.choice(len(p), p=p)
        next_word = loaded_tokenizer.index_word.get(next_idx, '')
        if not next_word:
            break
        generated.append(next_word)
        freq[next_word] = freq.get(next_word, 0) + 1

    return ' '.join(generated)

# Пример использования загруженной модели
print(generate_from_loaded("кот сидит", next_words=15))
```

### 5.13. Интерактивный режим (по желанию)

Вы можете добавить интерактивный цикл для ввода фраз и получения перевода как с текущей, так и с загруженной моделью.

---

## 6. Полный код нейронной сети с обучением

Для удобства ниже приведён сводный код всех ячеек, необходимых для полного цикла обучения модели (без интерактива). Вы можете скопировать его в одну ячейку или выполнять по частям.

```python
# Импорты
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import pickle
import random
import requests
from sklearn.model_selection import train_test_split

# Загрузка корпуса из репозитория (укажите вашу ссылку)
url_corpus = "https://raw.githubusercontent.com/ваш_логин/text-generation-lstm/main/corpus.txt"
response = requests.get(url_corpus)
with open('corpus.txt', 'w', encoding='utf-8') as f:
    f.write(response.text)

with open('corpus.txt', 'r', encoding='utf-8') as f:
    corpus_lines = [line.strip().lower() for line in f if line.strip()]
print(f"Загружено строк: {len(corpus_lines)}")

# Токенизация
text = ' '.join(corpus_lines)
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1
print(f"Размер словаря: {total_words}")

# Создание последовательностей
sequence_length = 5
tokens = tokenizer.texts_to_sequences([text])[0]
input_sequences = [tokens[i-sequence_length:i+1] for i in range(sequence_length, len(tokens))]
X = np.array([seq[:-1] for seq in input_sequences])
y = np.array([seq[-1] for seq in input_sequences])
print(f"Всего примеров: {len(X)}")

# Разделение
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Модель
model = Sequential([
    Embedding(total_words, 128, input_length=sequence_length),
    LSTM(256, return_sequences=True, dropout=0.2),
    LSTM(256, dropout=0.2),
    Dense(total_words, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Обучение
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=50, batch_size=64,
                    validation_data=(X_val, y_val), callbacks=[early_stop], verbose=1)

# Графики
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.title('Потери')
plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.legend()
plt.title('Точность')
plt.show()

# Сохранение
model.save('text_generator.h5')
with open('tokenizer.pickle', 'wb') as f:
    pickle.dump(tokenizer, f)
print("Модель и токенизатор сохранены.")
```

---

## 7. Код с загруженной моделью для быстрого запуска

Этот блок можно использовать отдельно, если модель уже сохранена в репозитории. Он загружает файлы по raw‑ссылкам и генерирует текст.

```python
# Загрузка из репозитория (укажите ваши ссылки)
url_model = "https://github.com/ваш_логин/text-generation-lstm/raw/main/text_generator.h5"
url_tokenizer = "https://github.com/ваш_логин/text-generation-lstm/raw/main/tokenizer.pickle"

!wget -O text_generator.h5 {url_model}
!wget -O tokenizer.pickle {url_tokenizer}

from tensorflow.keras.models import load_model
import pickle
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

loaded_model = load_model('text_generator.h5')
with open('tokenizer.pickle', 'rb') as f:
    loaded_tokenizer = pickle.load(f)

sequence_length = 5

def generate_quick(seed_text, next_words=20, temperature=0.8, top_k=10, rep_penalty=1.2):
    words = seed_text.lower().split()
    generated = words.copy()
    freq = {w: generated.count(w) for w in generated}
    for _ in range(next_words):
        current_seq = ' '.join(generated[-sequence_length:])
        seq = loaded_tokenizer.texts_to_sequences([current_seq])[0]
        seq = pad_sequences([seq], maxlen=sequence_length, padding='pre')
        preds = loaded_model.predict(seq, verbose=0)[0]

        logits = np.log(preds + 1e-8) / temperature
        exp_logits = np.exp(logits)
        p = exp_logits / exp_logits.sum()

        if top_k and top_k < len(p):
            top_indices = np.argsort(p)[-top_k:]
            mask = np.zeros_like(p)
            mask[top_indices] = 1
            p = p * mask
            p = p / p.sum()

        for idx, word in loaded_tokenizer.index_word.items():
            count = freq.get(word, 0)
            if count > 0 and idx < len(p):
                p[idx] = p[idx] / (rep_penalty ** count)
        p = p / p.sum()

        next_idx = np.random.choice(len(p), p=p)
        next_word = loaded_tokenizer.index_word.get(next_idx, '')
        if not next_word:
            break
        generated.append(next_word)
        freq[next_word] = freq.get(next_word, 0) + 1
    return ' '.join(generated)

print(generate_quick("кот сидит", next_words=15))
```

---

## 8. Задания для студентов

### Задание 1. Подготовка репозитория и данных
- Создайте публичный репозиторий на GitHub.
- Загрузите в него текстовый корпус (собственный или сгенерированный синтетический) с именем `corpus.txt`. Убедитесь, что объём корпуса соответствует требованиям (не менее 1000 строк).
- Получите raw‑ссылку на файл и запишите её для использования в ноутбуке.

### Задание 2. Подготовка данных в ноутбуке
- В ноутбуке напишите код для загрузки корпуса из репозитория по ссылке.
- Выведите статистику: количество строк, общее количество слов, размер словаря, среднюю длину предложения.
- Выполните токенизацию и создайте обучающие последовательности с длиной окна `sequence_length = 5`. Объясните, почему для обучения используются скользящие окна.
- Разделите данные на обучающую и валидационную выборки.

### Задание 3. Построение модели
- Постройте модель с двумя слоями LSTM (можно поэкспериментировать с количеством нейронов и dropout).
- Выведите summary модели.
- Скомпилируйте модель с оптимизатором 'adam' и функцией потерь 'sparse_categorical_crossentropy'.

### Задание 4. Обучение модели
- Обучите модель с использованием `EarlyStopping` (patience=5) на 50 эпохах.
- Постройте графики потерь и точности на обучающей и валидационной выборках.
- Проанализируйте, не переобучилась ли модель.

### Задание 5. Генерация текста (с текущей моделью)
- Реализуйте функцию `generate_text` с поддержкой температуры, top‑k и штрафа за повторения.
- Протестируйте генерацию для разных начальных фраз (3–5 примеров). Попробуйте разные значения температуры (0.5, 1.0, 1.5) и top‑k (3, 10, None). Опишите, как меняется качество и разнообразие текста.

### Задание 6. Сохранение и загрузка в репозиторий
- Сохраните обученную модель в файл `text_generator.h5`, а токенизатор – в `tokenizer.pickle`.
- Загрузите эти файлы в свой репозиторий на GitHub.
- Получите raw‑ссылки на оба файла (используйте формат `https://github.com/.../raw/main/...`).

### Задание 7. Загрузка модели из репозитория и быстрая генерация
- В отдельной ячейке напишите код, который скачивает файлы модели и токенизатора из репозитория по ссылкам.
- Загрузите модель и токенизатор, и с их помощью сгенерируйте текст для тех же начальных фраз, что и в задании 5. Убедитесь, что результаты сопоставимы (с учётом случайности).

### Задание 8. Анализ результатов
- Оцените качество сгенерированного текста визуально. В каких случаях модель генерирует бессмысленные или повторяющиеся фразы?
- Предложите способы улучшения модели (например, увеличение размера корпуса, использование большего контекста, добавление Attention, использование предобученных эмбеддингов).

### Задание 9*. Дополнительно (по желанию)
- Реализуйте генерацию на уровне символов (character‑level) вместо слов. Сравните результаты.
- Добавьте в модель механизм внимания.
- Попробуйте использовать двунаправленный LSTM в первом слое.

---

## 9. Заключение

В ходе выполнения этого задания вы:
- создали собственный репозиторий на GitHub и научились загружать туда файлы;
- познакомились с основными этапами создания модели генерации текста на основе LSTM;
- подготовили текстовые данные, создали обучающие последовательности;
- построили и обучили рекуррентную нейронную сеть;
- освоили методы сэмплирования (температура, top‑k, штраф за повторения) для получения разнообразного текста;
- научились сохранять модель и токенизатор, а затем загружать их из репозитория для повторного использования.

Полученные навыки являются фундаментом для понимания современных языковых моделей (GPT, LLaMA и др.), которые также используют авторегрессионный принцип генерации, но с более сложными архитектурами (трансформеры).

---

**Примечание:** Все ячейки кода можно запускать последовательно в Google Colab. Не забудьте подставить свои ссылки на файлы в репозитории. Убедитесь, что файлы загружены в репозиторий и доступны по прямым ссылкам. При возникновении проблем с памятью уменьшите размер словаря или количество эпох.